###
get the file path for the folder named: "Waiahole_FIM_EXAMPLE"

## create a function in this file (any function for example X + Y. then call that function in to the python file: "RUN_GSSHA_FIM" to make sure it works

In [1]:
import pandas as pd
from datetime import datetime
from pathlib import Path
import os
import re
import shutil
import subprocess
import math
from scipy.optimize import brentq

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
#from sklearn.metrics import r2_score
from scipy.optimize import fsolve



import numpy as np


from datetime import datetime, timedelta
from datetime import datetime, timezone
from dateutil.tz import tzutc, tzlocal

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

from scipy import stats

#import arcpy

In [2]:
def read_prj_file(GSSHA_prj_name, prj_folder_path):
    #read the prj file
    prj_path = prj_folder_path / GSSHA_prj_name 
    
    # Read lines, skipping the first two GSSHA header lines
    with open(prj_path, "r") as file:
        lines = file.readlines()[2:]
    
    # Parse lines into [Parameter, Value] lists
    rows = []
    
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue  # Skip empty lines
    
        parts = stripped.split(None, 1)  # split on first space(s)
        if len(parts) == 2:
            rows.append([parts[0], parts[1]])
        else:
            rows.append([parts[0], ""])
    
    # Create the DataFrame
    df_prj = pd.DataFrame(rows, columns=["Parameter", "Value"])
    df_prj = df_prj.set_index("Parameter")

    return df_prj

def convert_df_to_prj(df_prj, output_folder, prj_file_name):
    """
    Converts df_prj (with columns: Parameter, Value, [commented]) into a .prj config file for GSSHA.
    Adds GSSHA header at the top.

    Parameters:
    - df_prj (pd.DataFrame): DataFrame with at least 'Parameter' and 'Value' columns.
    - output_folder (str or Path): Folder where the .prj file will be saved.
    - prj_file_name (str): Name of the .prj file (without extension).
    """
    # Ensure output folder ex

    # Define output file path
    output_path = output_folder / f"{prj_file_name}.prj"

    # Open and write line-by-line
    with open(output_path, 'w', encoding='utf-8') as f:
        # Add GSSHA header
        f.write("GSSHAPROJECT\n")
        f.write("WMS WMS 11.1.10 (64-bit)\n")

        for _, row in df_prj.iterrows():
            key = str(row['Parameter']).strip()
            value = str(row['Value']).strip()
            
            # Optional 'commented' column
            is_commented = False
            if 'commented' in df_prj.columns:
                is_commented = str(row['commented']).strip().lower() in ['true', '1', 'yes', '#']

            # Add quotes if value looks like a file path
            if "\\" in value or "/" in value:
                value = f'{value}'

            line = f"{key:<25} {value}"
            if is_commented:
                line = f"#{line}"
            
            f.write(line + '\n')

    print(f"✅ Successfully wrote: {output_path}")
    return output_path


In [3]:
cwd = os.getcwd()
script_dir = Path.cwd()  # Use  if you're in Jupyter or interactive session
MODEL_DIR = script_dir / 'testing_files_FOR_FUNCTIONS'
GSSHA_prj_name = "Waiahole_FIM_Model"
df_prj = read_prj_file(GSSHA_prj_name +".prj", MODEL_DIR)

In [4]:
df_prj

,Value
Parameter,
WATERSHED_MASK,"""C:\Users\bgorberg\Documents\WMS_2025\Waiahole..."
PROJECT_PATH,"""C:\Users\bgorberg\Documents\WMS_2025\Waiahole..."
#LandSoil,"""C:\Users\bgorberg\Documents\WMS_2025\Waiahole..."
#PROJECTION_FILE,"""C:\Users\bgorberg\Documents\WMS_2025\Waiahole..."
NON_ORTHO_CHANNELS,
FLINE,"""C:\Users\bgorberg\Documents\WMS_2025\Waiahole..."
METRIC,
GRIDSIZE,10.000000
ROWS,107


## Fix the .prj file paths

In [9]:
def change_prj_filepaths_to_model_working_directory(working_model_folder = "testing_files_FOR_FUNCTIONS"):
    

_IncompleteInputError: incomplete input (2924498028.py, line 2)

In [5]:
import os
import re
from pathlib import Path


def fix_prj_paths(df_prj, model_dir):
    """
    Rewrite the file paths in a GSSHA .prj DataFrame so they point to THIS
    machine instead of the original author's computer.

    For every Value that contains a file path:
      1. checks whether the value holds a path,
      2. keeps only the last part (the filename, e.g. 'Waiahole_FIM_Model.msk'),
      3. rebuilds it inside `model_dir` (the testing_files_FOR_FUNCTIONS folder).

    Special cases handled automatically:
      - PROJECT_PATH ends in a separator (a folder, not a file) -> points at model_dir.
      - #INDEXGRID_GUID has a path AND a GUID after it -> only the path is rewritten.
    """
    model_dir = Path(model_dir)
    df = df_prj.copy()

    # Matches a Windows path (drive letter + separator) up to the next quote,
    # so a trailing GUID on the #INDEXGRID_GUID row is left untouched.
    windows_path = re.compile(r'[A-Za-z]:[\\/][^"]*')

    def rebuild(match):
        path = match.group(0).strip()
        if path.endswith('\\') or path.endswith('/'):     # folder, e.g. PROJECT_PATH
            return str(model_dir) + os.sep
        filename = path.replace('\\', '/').rstrip('/').split('/')[-1]
        return str(model_dir / filename)

    for i, row in df.iterrows():
        value = str(row['Value'])
        if '\\' in value or '/' in value:                 # this row holds a path
            df.at[i, 'Value'] = windows_path.sub(rebuild, value)

    return df

In [6]:
cwd = os.getcwd()
script_dir = Path.cwd() 

In [7]:
script_dir 

PosixPath('/Users/reimiyashiro/Documents/Flood_Stage_Maps/RUN_GSSHA')

In [8]:
from pathlib import Path

# Your cloned repo's RUN_GSSHA folder
BASE = Path('/Users/reimiyashiro/Documents/Flood_Stage_Maps/RUN_GSSHA')

# The folder holding the model files (.prj, .msk, .ele, ...)
MODEL_DIR = BASE / 'testing_files_FOR_FUNCTIONS'

# Quick sanity check so you know the folder name is right
print("MODEL_DIR exists:", MODEL_DIR.exists())
if MODEL_DIR.exists():
    for f in sorted(MODEL_DIR.iterdir()):
        print("  ", f.name)
else:
    print("Not found — here's what's actually inside RUN_GSSHA:")
    for f in sorted(BASE.iterdir()):
        print("  ", f.name)

MODEL_DIR exists: True
   Waiahole_FIM_Model.prj


## Apply the fix and write the corrected .prj back out

In [9]:
GSSHA_prj_name = "Waiahole_FIM_Model"
df_prj   = read_prj_file(GSSHA_prj_name + ".prj", MODEL_DIR)
df_fixed = fix_prj_paths(df_prj, MODEL_DIR)
df_fixed

,Value
Parameter,
WATERSHED_MASK,"""/Users/reimiyashiro/Documents/Flood_Stage_Map..."
PROJECT_PATH,"""/Users/reimiyashiro/Documents/Flood_Stage_Map..."
#LandSoil,"""/Users/reimiyashiro/Documents/Flood_Stage_Map..."
#PROJECTION_FILE,"""/Users/reimiyashiro/Documents/Flood_Stage_Map..."
NON_ORTHO_CHANNELS,
FLINE,"""/Users/reimiyashiro/Documents/Flood_Stage_Map..."
METRIC,
GRIDSIZE,10.000000
ROWS,107


In [11]:
# Write the corrected DataFrame back out as a .prj file.
# reset_index() is required: read_prj_file made 'Parameter' the INDEX,
# but convert_df_to_prj expects 'Parameter' as a COLUMN.
# Writing to a *_local name so the original Waiahole_FIM_Model.prj is preserved.
# Once you've checked the output, change the name back to "Waiahole_FIM_Model".
convert_df_to_prj(df_fixed.reset_index(), MODEL_DIR, "Waiahole_FIM_Model_local")


✅ Successfully wrote: /Users/reimiyashiro/Documents/Flood_Stage_Maps/RUN_GSSHA/testing_files_FOR_FUNCTIONS/Waiahole_FIM_Model_local.prj


PosixPath('/Users/reimiyashiro/Documents/Flood_Stage_Maps/RUN_GSSHA/testing_files_FOR_FUNCTIONS/Waiahole_FIM_Model_local.prj')